# 04 · Find the TopK SAE feature that best detects cyclones

**Run this after notebook 03, with the same Google Drive account. A T4 is sufficient.** Select Runtime → Run all. The defaults use the course sample and `monthly_2017_2018/topk_runs/topk_baseline/best.pt` from notebook 02.

The saved SAE already knows how to turn each 512-value GraphCast row into sparse feature values. It stays frozen. A **probe** is a small classifier that asks how well **one** of those features predicts the human cyclone label at that mesh node. We fit one logistic probe per feature on 2019 storms, choose the feature and cutoff on 2020 storms, then score the locked choice on 2021 storms. Notebook 02’s three reconstruction dates only selected the SAE checkpoint.

## 1. Connect Drive and load the checkpoint

In [1]:
# @title Connect Drive and load the project
import json
from google.colab import drive

import os
import subprocess
import sys
from pathlib import Path

# One immutable code version is shared by all five notebooks.
RELEASE = '2cf2b3a5e4357b0fff88ebf3bbc88c320e0e371b'
REPO = Path('/content') / ('ams691_project_' + RELEASE[:12])
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'init', '-q', str(REPO)], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--verify', 'HEAD'],
                      capture_output=True, text=True)
if head.stdout.strip() != RELEASE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', '--depth', '1',
                    'https://github.com/observerhan/graphcast-interpretability.git', RELEASE], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RELEASE], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'src'))

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/AMS691_GraphCast')
LABEL_RUN = DRIVE_PROJECT / 'cyclones_2019_2021'
DATA = REPO / 'data' / 'cyclone_2019_2021'


Mounted at /content/drive


In [2]:
# @title Select the saved run
COHORT = 'course'  # @param ["course", "all_guarded"]
SAE_RUN = 'topk_baseline'  # @param {type:"string"}
SAE_EXPERIMENT = 'monthly_2017_2018'  # @param {type:"string"}
from cyclone_probes import open_experiment, encode_split, fit_probes, select_feature
from cyclone_probes import save_selection, evaluate_selected, write_table
from cyclone_data import write_json
from IPython.display import display
import pandas as pd
import gc

checkpoint = DRIVE_PROJECT / SAE_EXPERIMENT / 'topk_runs' / SAE_RUN / 'best.pt'
model, device, manifest, y_by_time, identity = open_experiment(LABEL_RUN, DATA, COHORT, checkpoint)
OUT = LABEL_RUN / COHORT / 'probe_runs' / (SAE_RUN + '_' + identity['checkpoint_sha256'][:12])
print(f"Loaded best.pt from epoch {identity['checkpoint_epoch']}. Device: {device}.")
print(f"Dictionary: {identity['latent']} features; up to {identity['k']} active per node.")
print('Results folder:', OUT)

Using device: cuda
Loaded best.pt from epoch 19. Device: cuda.
Dictionary: 4096 features; up to 32 active per node.
Results folder: /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course/probe_runs/topk_baseline_e078d6f4b612


## 2. Fit the probes on 2019 storms

For each feature, the probe learns a slope and intercept in `p = sigmoid(slope × activation + intercept)`. The activation is the feature value after TopK. We give cyclone nodes and background nodes equal total weight in the training loss. The fit includes every node from every 2019 fitting snapshot. Sparse storage lets one training pass account for all the zero feature values.

We divide by each feature’s training maximum for numerical scaling and use a small fixed L2 penalty on the scaled slope. The saved slope and cutoff are converted to the original feature units.

In [3]:
# @title Fit one logistic classifier per frozen SAE feature
X_fit, y_fit, _ = encode_split('fit', LABEL_RUN, manifest, y_by_time, model, device)
print(f'Fitting on {len(y_fit):,} mesh rows, including {y_fit.sum():,} cyclone rows.')
weights = fit_probes(X_fit, y_fit)
del X_fit, y_fit
gc.collect()

Encoding fit:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting on 491,544 mesh rows, including 742 cyclone rows.


Fitting single-feature probes:   0%|          | 0/4096 [00:00<?, ?it/s]

645

## 3. Choose the feature and its cutoff on 2020 storms

F1 combines **precision** (of the nodes flagged, how many were labeled cyclone?) and **recall** (of the labeled cyclone nodes, how many were found?). We compare all distinct activation cutoffs, including zero, for each probe that learned a direction on the fitting data. A negative slope lets a low activation predict a cyclone. The score pools all validation nodes, including quiet maps. Ties within a feature favor fewer flagged nodes. Ties between features favor fewer false positives and then the lower feature index.

In [4]:
# @title Select and save the best feature using validation F1
X_val, y_val, _ = encode_split('validation', LABEL_RUN, manifest, y_by_time, model, device)
ranking = select_feature(X_val, y_val, weights)
save_selection(OUT, identity, weights, ranking)
winner = ranking[0]
display(pd.DataFrame(ranking[:5])[['feature','rule','activation_cutoff','precision','recall','f1']])
print(f"Selected zero-based feature {winner['feature']}: predict cyclone when its activation "
      f"{winner['rule']} {winner['activation_cutoff']:.8g}.")
del X_val, y_val
gc.collect()

Encoding validation:   0%|          | 0/8 [00:00<?, ?it/s]

Choosing feature and cutoff:   0%|          | 0/4096 [00:00<?, ?it/s]

,feature,rule,activation_cutoff,precision,recall,f1
0,3962,>=,0.030217,0.447674,0.140511,0.213889
1,3288,>=,0.083200,0.357895,0.124088,0.184282
2,3110,>=,0.066389,0.242857,0.093066,0.134565
3,3227,>=,0.029244,0.353982,0.072993,0.121029
4,3157,>=,0.051520,0.132743,0.109489,0.120000


Selected zero-based feature 3962: predict cyclone when its activation >= 0.030217404.


50

## 4. Evaluate the saved choice on 2021 storms

The test cell reads the feature ID and cutoff from disk and applies them as saved. The test F1 is your estimate for unseen storms in this sampled experiment. The proportions of positive and quiet snapshots were set in advance for the course run.

In [5]:
# @title Measure precision, recall and F1 on the held-out group
selection = json.loads((OUT / 'selected_feature.json').read_text())
if selection['experiment'] != identity:
    raise ValueError('The selection belongs to a different checkpoint or labeled dataset.')
winner = selection['validation']
X_test, y_test, centers = encode_split('test', LABEL_RUN, manifest, y_by_time, model, device)
test, by_time = evaluate_selected(X_test, y_test, centers, winner)
write_json(OUT / 'test_metrics.json',
           dict(experiment=identity, feature=winner['feature'], rule=winner['rule'],
                activation_cutoff=winner['activation_cutoff'], test=test))
write_table(OUT / 'test_by_time.csv', by_time)
fields = ['tp','fp','fn','tn','precision','recall','f1']
display(pd.DataFrame([{k: s[k] for k in fields} for s in (winner, test)],
                     index=['2020 validation (chosen here)', '2021 test (locked choice)']))
print('Feature, rule and validation result:', OUT / 'selected_feature.json')
print('Held-out result:', OUT / 'test_metrics.json')
del X_test, y_test
gc.collect()

Encoding test:   0%|          | 0/8 [00:00<?, ?it/s]

,tp,fp,fn,tn,precision,recall,f1
2020 validation (chosen here),77,95,471,327053,0.447674,0.140511,0.213889
2021 test (locked choice),132,160,428,326976,0.452055,0.235714,0.309859


Feature, rule and validation result: /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course/probe_runs/topk_baseline_e078d6f4b612/selected_feature.json
Held-out result: /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course/probe_runs/topk_baseline_e078d6f4b612/test_metrics.json


42

`fitted_probes.csv` contains all slopes and intercepts. `validation_ranking.csv` shows how the candidate features ranked. `test_by_time.csv` lets you inspect hits, misses and false alarms on each 2021 map. Feature numbers belong to the checkpoint that produced them. To compare a different SAE, use the same labeled times, splits and metrics.

[MacMillan & Ouellette, single-feature probes](https://arxiv.org/html/2512.24440v1#S2.SS5) · [ClimateNetLarge labeling methods](https://doi.org/10.1038/s41597-025-05480-0)